# Passo 3: Operações DML no Delta Lake
Aplicamos INSERT, UPDATE e DELETE na tabela Delta na camada Bronze.

In [2]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

# Inicializa o Spark
spark = SparkSession.builder \
    .appName("DML-Delta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

bronze_path = "s3a://bronze/clientes_delta"
delta_table = DeltaTable.forPath(spark, bronze_path)

print("Dados Iniciais:")
delta_table.toDF().show()

# 1. UPDATE: Atualizar idade do cliente de ID 1
print("Executando UPDATE...")
delta_table.update(
    condition="id = 1",
    set={"idade": "99"}
)

# 2. DELETE: Remover cliente de ID 3
print("Executando DELETE...")
delta_table.delete(condition="id = 3")

# 3. INSERT (Append): Adicionar novo cliente
print("Executando INSERT...")
novos_clientes = spark.createDataFrame([(4, "Joao", "joao@email.com", 25)], "id int, nome string, email string, idade int")


print("Dados Finais após DML:")
delta_table.toDF().show()

print("Histórico de versões (Time Travel):")
delta_table.history().select("version", "operation", "timestamp").show(truncate=False)


Dados Iniciais:


+---+------------+----------------+-----+
| id|        nome|           email|idade|
+---+------------+----------------+-----+
|  1|   Ana Silva|   ana@email.com|   99|
|  2|Carlos Souza|carlos@email.com|   35|
+---+------------+----------------+-----+

Executando UPDATE...
Executando DELETE...


Executando INSERT...
Dados Finais após DML:
+---+------------+----------------+-----+
| id|        nome|           email|idade|
+---+------------+----------------+-----+
|  1|   Ana Silva|   ana@email.com|   99|
|  2|Carlos Souza|carlos@email.com|   35|
+---+------------+----------------+-----+

Histórico de versões (Time Travel):
+-------+---------+-------------------+
|version|operation|timestamp          |
+-------+---------+-------------------+
|3      |UPDATE   |2026-05-07 11:41:56|
|2      |DELETE   |2026-05-07 11:39:30|
|1      |UPDATE   |2026-05-07 11:39:23|
|0      |WRITE    |2026-05-07 11:15:23|
+-------+---------+-------------------+

